# NodeSubstrates - CORA Citation Network

This notebook demonstrates NodeSubstrates on the **CORA citation network** (papers as nodes, citations as edges).

**Note:** The CORA dataset file was separately downloaded and post-processed; Node2Vec embeddings were computed offline and attached to each node under the `embedding` attribute (this notebook reads the prepared JSON from `data/cora_network.json`).

**Use Case:** Explore topical communities, influential/bridge papers, and compare structural embeddings vs. textual bag-of-words features.

In [4]:
from node_substrates import NodeSubstratesWidget
from node_substrates.datasets.coranetwork import load_cora_network

## 1. Dataset Overview

Summary

The document includes:

**Node Attributes:**

- `id` (0–2707): Paper identifier
- `features` (1433 binary values): Bag-of-words vectors (word present=1, absent=0)
- `label` (0–6): Research topic class
- `class_name`: Human-readable topic (Neural Networks, Theory, etc.)

**Edge Attributes:**

- `source` & `target`: Citation relationship (citing → cited)

**Class Distribution:**

- Neural Networks: 30.2% (818 papers)
- Genetic Algorithms: 15.4% (418 papers)
- Probabilistic Methods: 15.7% (426 papers)
- Case Based: 11.0% (298 papers)
- Theory: 13.0% (351 papers)
- Reinforcement Learning: 8.0% (217 papers)
- Rule Learning: 6.6% (180 papers)

**Feature Details:**

- 1,433-word vocabulary (after stemming + stopword removal)
- Only words appearing in ≥10 documents included
- Binary representation (presence/absence, not frequency)

**Note:** The file was separately downloaded and post-processed; Node2Vec embeddings are attached to nodes under the `embedding` attribute.

In [5]:
# Load the CORA network
G = load_cora_network()

# Select only 100 nodes for debugging/DEVELOPMENT
# nodes_subset = list(G.nodes())[:100]
# G    = G.subgraph(nodes_subset).copy()

print(f"\nNetwork: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Class distribution by `class_name` or `subject`
class_counts = {}
for node, attrs in G.nodes(data=True):
    cls = attrs.get('class_name')  or 'Unknown'
    class_counts[cls] = class_counts.get(cls, 0) + 1

print("\nClass distribution:")
for cls, cnt in sorted(class_counts.items(), key=lambda x: -x[1]):
    print(f"  {cls}: {cnt}")

# Sample node attributes (show embedding size if present)
print("\nSample node attributes:")
sample_node = list(G.nodes())[0]
for key, value in G.nodes[sample_node].items():
    if key == 'embedding' and hasattr(value, '__len__'):
        print(f"  {key}: (len={len(value)})")
    else:
        print(f"  {key}: {value}")

Loaded CORA network: 2708 nodes, 5278 edges
Subjects: {'Unknown': 2708}

Network: 2708 nodes, 5278 edges

Class distribution:
  Unknown: 2708

Sample node attributes:
  features: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 

## 2. Create NodeSubstrates Widget

The initial view shows a force-directed layout revealing topical communities and citation structure.

Use the hybrid view to compare textual bag-of-words features, structural Node2Vec embeddings, and graph-derived metrics (degree, betweenness).

In [7]:
# Create widget with zoomed out view for better layout spread
# initial_scale=0.3 computes layout for ~3x larger area in each dimension
widget = NodeSubstratesWidget(G, auto_substrate=False, width=1200, height=1000, initial_scale=0.40)

# Interaction hints:
# - Pan: drag on empty space
# - Zoom: scroll wheel
# - Lasso select: Shift+drag (popup appears automatically)
# - Move substrate: Ctrl+drag on substrate region
# - Context menu: right-click on node or substrate
widget

Layout 'spring' computed in 320.68s for 2708 nodes


## 3. Auto-detected Substrate Suggestions

NodeSubstrates may suggest regions such as:

- **Influential papers (high-degree)**: Frequently cited works that act as hubs
- **Topical clusters**: Groups of papers sharing similar bag-of-words features or class labels
- **Bridging papers**: Documents connecting otherwise separate topic communities (high betweenness)
- **Embedding neighborhoods**: Papers close in Node2Vec embedding space indicating structural similarity

In [9]:
print("Auto-detected substrate suggestions:\n")
for i, suggestion in enumerate(widget.suggested_regions[:5]):  # Show top 5
    print(f"Suggestion {i}: {suggestion['label']}")
    print(f"  Nodes: {len(suggestion['node_ids'])}")
    print(f"  Score: {suggestion['score']:.3f}")
    print(f"  Reason: {suggestion['reason']}")
    print(f"  Recommended DR: {suggestion['recommended_dr']}")
    print()

Auto-detected substrate suggestions:

Suggestion 0: Community 74
  Nodes: 169
  Score: 0.827
  Reason: 169 nodes with high attribute diversity, similar nodes scattered in topology, potential outliers among neighbors
  Recommended DR: umap

Suggestion 1: Community 6
  Nodes: 201
  Score: 0.825
  Reason: 201 nodes with high attribute diversity, similar nodes scattered in topology, potential outliers among neighbors
  Recommended DR: umap

Suggestion 2: Community 81
  Nodes: 182
  Score: 0.746
  Reason: 182 nodes with high attribute diversity, similar nodes scattered in topology, potential outliers among neighbors
  Recommended DR: umap



## 4. Create a Substrate for High-Connectivity Nodes

Let's create a substrate focusing on the most connected professionals 
(lawyers and doctors) to see if they cluster by behavior patterns.

In [ ]:
# Select high-degree papers (influential / highly-cited)
high_degree = [
    str(node) for node, attrs in G.nodes(data=True)
    if attrs.get('degree', 0) >= 10
]

print(f"Found {len(high_degree)} high-degree papers")

if len(high_degree) >= 3:
    substrate_id = widget.create_substrate(
        high_degree,
        dr_method='umap',
        label='High-Degree Papers'
    )
    print(f"Created substrate: {substrate_id}")

## 5. Compare DR Methods

Different dimensionality reduction methods reveal different patterns:
- **PCA**: Linear relationships (e.g., degree vs betweenness)
- **UMAP**: Non-linear topical clusters (good for discovering communities)
- **t-SNE**: Local neighborhood emphasis (useful for embedding neighborhoods)

You can also compare Node2Vec structural embeddings with textual bag-of-words features within substrates.

In [ ]:
# Try UMAP for cluster discovery
if widget.substrates:
    widget.update_dr_method(widget.substrates[0]['id'], 'umap')
    print("Switched to UMAP - look for clusters of similar behavior patterns")

## 6. Interactive Exploration

Use lasso selection and substrates to investigate topical clusters and influential papers:

- **Shift+Drag** to select nodes
- Create substrates from selections to compare neighborhoods
- Compare Node2Vec neighborhoods vs. bag-of-words clustering

In [ ]:
# Check current selection
print(f"Selected nodes: {widget.selected_nodes}")

# Create substrate from selection (need at least 3 nodes)
if len(widget.selected_nodes) >= 3:
    substrate_id = widget.create_substrate(
        widget.selected_nodes,
        dr_method='umap',
        label='Investigation Selection'
    )
    print(f"Created substrate from selection: {substrate_id}")

## 7. Current Substrates

In [ ]:
# List current substrates
print("Current substrates:")
for s in widget.substrates:
    print(f"  {s['id']}: {s['label']} ({len(s['node_ids'])} nodes, {s['dr_method']})")

In [ ]:
# Dissolve a substrate to return nodes to force-directed layout
if widget.substrates:
    substrate_to_dissolve = widget.substrates[0]['id']
    widget.dissolve_substrate(substrate_to_dissolve)
    print(f"Dissolved {substrate_to_dissolve}")

## 8. CORA Network Insights

The hybrid NodeSubstrates view helps researchers by:

**Network View (Force-Directed):**
- Reveals topical communities connected by citation links
- Shows which papers act as hubs (high in-degree) or bridges (high betweenness)

**Attribute / Embedding View (Substrates):**
- Groups papers by textual features or Node2Vec embeddings
- Highlights outliers with unusual topical signatures
- Enables side-by-side comparison of embedding neighborhoods and class labels

**Key analysis points:**
- Highly-cited (hub) papers per topic
- Papers that bridge topics and may be multi-disciplinary
- Differences between textual-similarity clusters and structural-embedding neighborhoods

In [ ]:
# Summary statistics
print(f"\n=== CORA Network Summary ===")
print(f"Total nodes: {len(widget.nodes)}")
print(f"Nodes in substrates: {len(widget.substrate_node_ids)}")
print(f"Nodes in force-directed: {len(widget.topological_node_ids)}")
print(f"Active substrates: {len(widget.substrates)}")

# Class breakdown
print(f"\n=== Class Distribution ===")
for cls, count in sorted(class_counts.items(), key=lambda x: -x[1]):
    print(f"  {cls}: {count}")